# enrich.ipynb
**Step 2 of 3 — Feature Enrichment**

Joins synthetic cardholder and merchant attributes to each cleaned transaction row, producing an enriched dataframe ready for MongoDB ingestion.

> **Note:** Synthetic fields are generated deterministically from a seeded RNG — they are NOT real cardholder data.

**Run order:** `ingest.ipynb` → `enrich.ipynb` → `load_mongo.ipynb`

## 1. Imports & Logging Setup

In [ ]:
import logging
import numpy as np
import pandas as pd

# Configure logging to both file and notebook output
logging.basicConfig(
    filename="enrich.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

print("Imports complete.")

## 2. Configuration

In [ ]:
INPUT_PATH = "cleaned.csv"
OUTPUT_PATH = "enriched.csv"

# Merchant Category Codes and regions for synthetic merchant assignment
MCC_POOL = ["5411", "5812", "4111", "5999", "7011", "5912", "4814", "5651"]
REGION_POOL = ["Western Europe", "Northern Europe", "Southern Europe", "Eastern Europe"]

# Seed for reproducibility
RNG_SEED = 42

print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"MCC pool size: {len(MCC_POOL)} | Region pool size: {len(REGION_POOL)}")

## 3. Load Cleaned Data

In [ ]:
def load_cleaned(path: str) -> pd.DataFrame:
    """Load the cleaned dataframe produced by ingest.ipynb."""
    logging.info(f"Loading cleaned data from {path}")
    try:
        df = pd.read_csv(path)
        logging.info(f"Loaded {len(df)} rows.")
        return df
    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        raise

df = load_cleaned(INPUT_PATH)
print(f"Shape: {df.shape}")
df.head()

## 4. Add Transaction IDs

In [ ]:
def add_transaction_id(df: pd.DataFrame) -> pd.DataFrame:
    """Add a unique transaction_id to each row based on its index."""
    logging.info("Adding transaction IDs...")
    df = df.reset_index(drop=True)
    df["transaction_id"] = [f"txn_{i:06d}" for i in df.index]
    return df

df = add_transaction_id(df)
print(f"Sample IDs: {df['transaction_id'].head(3).tolist()}")

## 5. Add Synthetic Cardholder Features

In [ ]:
def add_cardholder_features(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """
    Add synthetic cardholder sub-document fields:
    - cardholder_account_age_days : randomly sampled 30–3650 days
    - cardholder_avg_monthly_spend: estimated from Amount with added noise
    - cardholder_velocity_7d      : randomly sampled 1–50 transactions
    """
    logging.info("Adding synthetic cardholder features...")
    n = len(df)

    df["cardholder_account_age_days"] = rng.integers(30, 3651, size=n)

    # Base avg monthly spend on transaction Amount with Gaussian noise
    df["cardholder_avg_monthly_spend"] = (
        df["Amount"] * rng.uniform(0.8, 20.0, size=n)
    ).round(2).clip(lower=1.0)

    df["cardholder_velocity_7d"] = rng.integers(1, 51, size=n)

    logging.info("Cardholder features added.")
    return df

rng = np.random.default_rng(RNG_SEED)
df = add_cardholder_features(df, rng)

print("Cardholder feature summary:")
print(df[["cardholder_account_age_days", "cardholder_avg_monthly_spend", "cardholder_velocity_7d"]].describe().round(2))

## 6. Add Synthetic Merchant Features

In [ ]:
def add_merchant_features(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """
    Add synthetic merchant sub-document fields:
    - merchant_mcc      : randomly sampled from MCC_POOL
    - merchant_region   : randomly sampled from REGION_POOL
    - merchant_id       : deterministic string ID based on random integer
    """
    logging.info("Adding synthetic merchant features...")
    n = len(df)

    df["merchant_mcc"] = rng.choice(MCC_POOL, size=n)
    df["merchant_region"] = rng.choice(REGION_POOL, size=n)
    df["merchant_id"] = [f"merch_{i:05d}" for i in rng.integers(1, 10001, size=n)]

    logging.info("Merchant features added.")
    return df

df = add_merchant_features(df, rng)

print("MCC distribution:")
print(df["merchant_mcc"].value_counts())
print()
print("Region distribution:")
print(df["merchant_region"].value_counts())

## 7. Preview Enriched Data

In [ ]:
print(f"Enriched shape: {df.shape}")
print(f"New columns added: transaction_id, cardholder_account_age_days, cardholder_avg_monthly_spend, cardholder_velocity_7d, merchant_mcc, merchant_region, merchant_id")
df[["transaction_id", "Amount", "Class", "cardholder_account_age_days",
    "cardholder_velocity_7d", "merchant_mcc", "merchant_region"]].head()

## 8. Save Enriched Data

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
logging.info(f"Enriched data saved to {OUTPUT_PATH} with {len(df.columns)} columns.")
print(f"Done. {len(df)} rows saved to '{OUTPUT_PATH}'.")
print("Next step: run load_mongo.ipynb")